# UW Quantum Machine Learning Hackathon - Quantum Data Encoding: Expectation, Reality, and Beyond

<div class="alert alert-success">

<h2>At a Glance</h2>

- **You need**: Python, pandas, scikit-learn, matplotlib
- **You do not need**: Quantum mechanics
- **Required**: Track 0 (baseline)
- **Then choose**: At least one experiment from Track 1 or Track 2
- **Deliverable**: 10-minute team presentation
- **Goal**: Make one evidence-based claim about whether, when, and why quantum encoding helps

**Minimum viable success:** 

Complete Track 0, run one well-designed experiment from Track 1 or Track 2, and make one evidence-based claim in your presentation. That is a good hackathon outcome.

</div>

## Welcome!

Machine learning models don't understand data directly — they understand *geometry*. Every encoding of data defines a notion of distance, similarity, and structure. Change the encoding, and you change what the model can learn.

A quantum computer offers a fundamentally new way to encode data. But "new" does not automatically mean "better." This hackathon is a direct inquiry into that question: **when does quantum data encoding actually help, and when doesn't it?**

You will work with pre-computed quantum projections generated from IBM hardware, applied to a cancer immunotherapy design problem. Your job is not to just reuse the data — your job is to **test it** and evaluate whether a quantum encoding benefit actually appears in this setting.


### What Makes This Different

- **No quantum physics required** — Focus on data science and experimental thinking
- **Real quantum hardware data** — Pre-computed results from IBM Heron R2
- **Open-ended exploration** — No "right" answers, only evidence-based discoveries
- **Research mindset** — 💡 Scientist mode ON! - curious, questioning, and unafraid to explore

### How to Start

1. **Read this notebook first.** Understand the problem, the methodology, and the challenge tracks.
2. **Open the companion tutorial notebook.** It contains all the working code, data loading, and helper functions.
3. **Choose your track(s).** You may focus on one track deeply or attempt multiple.
4. **Document everything.** Your presentation should follow the Expectation → Validation → Limitation → Attempt framework described in Section 4.

<div class="alert alert-info">

**This notebook is a guide.** 

It contains no code. Do your coding in the companion tutorial notebook.
</div>

## 1. The Problem: CAR T-Cell Cytotoxicity Prediction

### The Biology

**CAR T-cell therapy** is a personalized cancer treatment where a patient's immune cells (T-cells) are genetically engineered to express chimeric antigen receptors (CARs) that target specific proteins on cancer cells. These modified T-cells can recognize and destroy cancer cells more effectively.

Each CAR T-cell's effectiveness depends on its "recipe" — which **signaling motifs** (M1–M13) are placed in specific positions within the CAR's intracellular domain, plus a terminal marker (M14). The motifs are structural or functional components that recruit different signaling proteins and determine how the T-cell activates its immune response. Based on these motifs, our task is to predict the **cytotoxicity** of a given CAR T-cell — how effectively it kills cancer cells.

### The Data

| Property | Value |
|----------|-------|
| Constructs tested | 246 (out of ~2,500 possible) |
| Positions per construct | 4 encoded positions (see note below) |
| Values per position | 15 (13 motifs + terminal + empty) |
| Task | **Binary classification**: high vs. low cytotoxicity |
| Threshold | Nalm6 survival threshold = 0.62 (cutoff value, see note below) |
| Split | 172 training, 74 test |

The experimental data comes from [Daniels et al. (2022)](https://doi.org/10.1126/science.abq0225), who built and screened a combinatorial library of CAR T-cell constructs. The quantum kernel analysis is based on [Utro et al. (2025)](https://arxiv.org/abs/2507.22710), and the companion tutorial notebook follows the [Qiskit PQK tutorial](https://quantum.cloud.ibm.com/docs/en/tutorials/projected-quantum-kernels).

**What does the threshold mean?** 

This is a **binary classification** task. The cytotoxicity measurement is the fraction of Nalm6 cancer cells that survive after exposure to the CAR T-cells. We use **0.62** as a cutoff value to divide constructs into two classes: high cytotoxicity (Nalm6 survival < 0.62, meaning the CAR T-cell killed most cancer cells) and low cytotoxicity (Nalm6 survival > 0.62).

**Why 4 positions?** 

The original data has 5 position columns. The tutorial drops the last one (which is always the M14 terminal motif) and uses encoding. Each position can hold any of the 15 classes (13 functional motifs, terminal, or empty). In practice, later positions are more likely to be empty or terminal, especially position 4.

**Note:** This is a research benchmark for studying representation learning, not a clinical decision tool.

### Why This Problem?

This is a **sparse data regime** — only ~10% of possible combinations have been tested. Theory suggests this is one setting where quantum encoding might help: classical ML benefits from more data, while quantum encoding can create different similarity structures. The gap between them is most visible when data is scarce.


## 2. The Methodology: Two Pipelines, One Question

Now that you understand the data, here is how the experiment works.

### The Experimental Setup

You will compare two ML pipelines that differ in **only one step** — the data encoding:

| Step | Pipeline A: Classical | Pipeline B: Quantum-Projected (PQK) |
|------|----------------------|-----------------------------|
| Input | Motif IDs (categorical) | Motif IDs (categorical) |
| Encoding | One-hot → 60-dim vector | One-hot → 60-qubit quantum circuit → measure → 180-dim vector |
| Classifier | RBF-SVM with GridSearchCV | RBF-SVM with GridSearchCV |
| Evaluation | Weighted F1 on test set | Weighted F1 on test set |

This isolation is what makes your experiments scientifically meaningful. Any performance difference must come from the encoding.

### Key Concepts

**Kernel Methods & Similarity**

A kernel function measures how "similar" two data points are. The RBF kernel used in this hackathon says: two points are similar if they are close in feature space, and dissimilar if they are far apart. The SVM classifier then draws boundaries based on this similarity — so the encoding decides what "close" means, and the SVM just follows.

**Projected Quantum Kernel (PQK)**

Think of the quantum circuit as a **nonlinear feature extractor** — it transforms the input data into a new representation where the relationship between features has been reshaped, somewhat like how hidden layers in a neural network transform inputs before a final classifier. [[Huang et al. (2021)](https://doi.org/10.1038/s41467-021-22539-9)][[Glick et al. (2024)](https://www.nature.com/articles/s41567-023-02340-9)]

Here is what it does, step by step:

1. Start with the same 60-dimensional one-hot vector used by the classical pipeline.
2. Load it into a 60-qubit quantum circuit (called a [ZZ Feature Map](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.ZZFeatureMap)), which creates entanglement between qubits — meaning the qubits influence each other in ways that depend on the input data.
3. Measure each qubit along three axes (called X, Y, and Z). Each measurement gives a number between −1 and +1. These three numbers capture the qubit's state from the circuit. (For the precise definition of these measurements and the underlying reduced density matrix, see the Appendix.)
4. Collect the results: 60 qubits × 3 measurements = 180-dimensional projected feature vector.

This 180-dim vector is then fed into a standard scikit-learn SVM, exactly the same way as the classical 60-dim vector. The quantum part is already done for you — the pre-computed projections are in the data files.


**Geometric Difference — Does the Quantum Encoding See Something Different?**

Huang et al. (2021) defined a quantity called the **geometric difference** $g$ that measures how differently two encodings organize data:

- **$g$ close to 1:** the classical and quantum encodings create similar notions of similarity. Classical ML should be competitive.
- **$g$ close to $\sqrt{N}$** (where $N$ is the training set size): the encodings are genuinely different. The quantum encoding may provide an advantage. Verify this on the actual task.

Not all datasets benefit from PQK. More data narrows any quantum encoding benefit, so the benefit is most likely in **data-scarce** settings like this one.

<div class="alert alert-info">

**Terminology:** 

Throughout this document, "quantum encoding," "quantum-projected features," and "PQK features" all refer to the same thing — the 180-dimensional vectors produced by the quantum circuit.

**Why "projected"?** 

The raw quantum state lives in an astronomically large space ($2^{60}$ dimensions). If you tried to measure similarity directly in that space, all data points would look equally far apart — useless for classification. The projection step collapses this back to a manageable 180-dim space while preserving the nonlinear features created by entanglement.

</div>

### Further Reading

- [IBM Quantum Learning: Data encoding of Quantum machine learning course](https://quantum.cloud.ibm.com/learning/en/courses/quantum-machine-learning/data-encoding)
- Huang et al. (2021). *[Power of data in quantum machine learning.](https://doi.org/10.1038/s41467-021-22539-9)*
- Glick et al. (2024). *[Covariant quantum kernels for data with group structure.](https://www.nature.com/articles/s41567-023-02340-9)*



## 3. Challenge Tracks

With the baseline established, it is time to investigate. There are **two** challenge tracks below, each probing a different aspect of quantum data encoding. You may focus on one track deeply or attempt multiple. Every track builds on a shared baseline (Track 0).

### Track 0: Establish the Baseline (Required)

Before investigating any track, you need a working baseline. This is **required for all teams.**

#### Your Challenge

Using the tutorial notebook provided for this hackathon (adapted from the [IBM PQK tutorial](https://quantum.cloud.ibm.com/docs/en/tutorials/projected-quantum-kernels)):

1. Load and preprocess the CAR T-cell data (one-hot encoding, π/2 scaling)
2. Train an RBF-SVM on the classical 60-dim features. Report the weighted F1 on the test set.
3. Train an RBF-SVM on the pre-computed 180-dim quantum projections. Report the weighted F1.
4. Visualize both kernel matrices as heatmaps (sorted by label). Check whether any label-sorted block structure appears.

#### What to record

- Classical F1 and quantum-projected F1
- The difference ΔF1 between them
- Whether the kernel heatmaps show visibly different structure


<div class="alert alert-info">

**Expected ballpark:** 

Both F1 values should fall in the 0.65–0.85 range. The gap is typically small (0–0.05). A small gap is normal. Investigate why it exists and when it changes.

**Tip:** 

Use cross-validation on the training set during exploration. Treat the held-out test set as a final confirmation, not a tuning target.
</div>

#### Bonus

Compute the geometric difference $g$ between the classical and quantum-projected kernel matrices:

$$g_{cq} = \sqrt{\|\sqrt{K^q}\, \sqrt{K^c}\, (K^c + \lambda I)^{-2}\, \sqrt{K^c}\, \sqrt{K^q}\|_{\infty}}$$

Check whether $g$ is closer to 1 or to $\sqrt{N}$ (where $N$ is the training set size). This tells you whether the two encodings see the data differently.


### Track 1: Anatomy of the Quantum Encoding

This track investigates **what parts of the quantum encoding matter.** The 180-dimensional projection is not a black box — it has internal structure (3 measurement bases × 4 motif positions). Your challenge is to take it apart and find out which pieces carry the signal.

#### Your Challenge

Design and run experiments that answer one or more of the following:

**A. Which measurement basis matters most?**

Train SVMs using only the ⟨X⟩ features (columns 0–59), only ⟨Y⟩ (60–119), or only ⟨Z⟩ (120–179). Compare their F1 scores to the full 180-dim model. Is the information concentrated in one basis, or distributed across all three?

**B. Where does the quantum encoding help most?**

Train both classical and quantum-projected SVMs using features from single positions (1, 2, 3) and from combinations (1+2, 1+2+3, all four). Use the projection column layout in the Appendix for slicing. Where is the quantum encoding benefit largest? Compare the performance gain across positions.

<div class="alert alert-info">

**Context from the literature:**

One prior study observed stronger effects in position 3 (the most data-sparse position). Use this as a hypothesis to test.

Interpret position 4 carefully: in this encoding it often reflects terminal or empty-slot structure, not just an additional motif-bearing position.

</div>

**C. Can you achieve similar performance with fewer features?**

Try feature selection or dimensionality reduction on the 180-dim projections. (Fit any selection or reduction step on the training set only, then apply it to the test set.) How much can you compress the quantum features before performance degrades? What does this tell you about the effective dimensionality of the quantum encoding?

#### Questions to answer

- Is the quantum encoding's value concentrated in specific components, or is it a distributed effect?
- Does the performance gain concentrate where data is scarce (e.g., position 3)?
- What is the minimum representation that preserves the quantum encoding's benefit?


### Track 2: Data Scarcity and Robustness

This track investigates **when the quantum encoding helps.** Theory predicts that quantum encodings are most useful in data-scarce regimes. Your challenge is to test this prediction — and to probe the encoding's robustness.

#### Your Challenge

Design and run experiments that answer one or more of the following:

**A. Learning curves: does data scarcity widen the gap?**

Subsample the training set to 25%, 50%, and 75% (maintaining class balance). For each fraction, repeat with 3–5 random seeds and report mean ± standard deviation. Plot F1 vs. training size with error bars. Does the quantum encoding benefit grow when data is scarce and shrink when data is plentiful?

**B. Robustness: how much noise can the encoding tolerate?**

Add Gaussian noise of increasing variance to the quantum projections (this is a classical operation on the existing data files). At what noise level does the quantum encoding benefit vanish? If you compare to adding noise to classical features, note that the two representations have different scales — compare relative degradation rather than raw noise magnitude, or standardize both before adding noise.

**C. Combinatorial structure: does the motif arrangement matter?**

Randomly permute motif assignments within each position column across **training samples only** (keep the test set unchanged). This preserves the marginal frequency of each motif at each position but destroys the combinatorial relationships between positions. Train the classical SVM on the shuffled training data and evaluate on the original test set. If performance drops, the task depends on inter-position structure — not just per-position category counts.

**D. (optional) Real hardware noise: how do fresh QPU results compare to the pre-computed projections?**

(Require QPU access)

Pick 10-40 training data points and run the ZZFeatureMap circuit on the QPU using the tutorial's single-datapoint code. Compare your fresh results to the corresponding rows in `projections_train.csv`. How close are they? If you also ran Experiment B, compare the magnitude of real hardware noise to your synthetic noise levels - does your artificial noise experiment reflect reality?

<div class="alert alert-info">

**Note:** 

This shuffle experiment helps you understand whether the task itself has combinatorial structure, which informs interpretation of any quantum encoding benefit. However, it only tells you about the *task structure*, not about the quantum encoding directly (because you cannot re-run the quantum circuit on shuffled data). Interpret these results conservatively, as the shuffle only tests task structure.

</div>

#### Questions to answer

- At what training size does the classical model catch up to the quantum-projected model?
- Is the quantum encoding more or less robust to noise than the classical encoding?
- Does destroying combinatorial structure hurt classification? What does that imply about what the quantum circuit might be capturing?



## 4. Your Final Presentation

At the end of coding, each team will present their findings in **10 minutes.** Structure your presentation around four questions:

### 1. Expectation

**What did you expect before running experiments?**

- What benefits, if any, did you expect quantum encoding to provide?

- Why? Based on what you read, heard, or reasoned about?

- State a clear hypothesis.

### 2. Validation

**What did you actually observe?**

- Which of your expectations were confirmed?

- What performance differences did you measure?

- Show specific plots, numbers, and evidence.

### 3. Limitation

**What didn't work?**

- Where did quantum encoding fail to help or even hurt?

- What constraints did you encounter?

- Why do these limitations exist?

### 4. Attempt

**How did you try to overcome the limitations?**

- What creative experiments did you design?

- What were the results?

- What would you try next with more time or resources?

### Presentation Tips

- **Be honest:** Negative results are as valuable as positive ones.
- **Show data:** Every claim should have supporting evidence.
- **Tell a story:** Take us through your journey of discovery.
- **Be critical:** Question assumptions, including your own.
- **Think forward:** What are the implications for quantum ML research?

## References

1. Utro, F. et al. (2025). *Enhanced Prediction of CAR T-Cell Cytotoxicity with Quantum-Kernel Methods.* [arXiv:2507.22710](https://arxiv.org/abs/2507.22710)
2. Huang, H.-Y. et al. (2021). *Power of data in quantum machine learning.* Nature Communications, 12, 2631. [DOI](https://doi.org/10.1038/s41467-021-22539-9)
3. Glick, J. R. et al. (2024). *Covariant quantum kernels for data with group structure.* Nature Physics, 20, 479–483. [DOI](https://doi.org/10.1038/s41567-023-02340-9)
4. Daniels, K. G. et al. (2022). *Decoding CAR T cell phenotype using combinatorial signaling motif libraries and machine learning.* Science, 378, 1194–1200. [DOI](https://doi.org/10.1126/science.abq0225)
5. [Enhance feature classification using projected quantum kernels](https://quantum.cloud.ibm.com/docs/en/tutorials/projected-quantum-kernels)


*Good luck! Test assumptions. Learn why.*


## Appendix: Mathematical Details

> **This section is optional.** It provides details for teams who want to compute the geometric difference (Track 0 Bonus) or understand the kernel formulas more deeply. Everything above is sufficient to start.

### The RBF Kernel

$$K(x, x') = \exp(-\gamma \|x - x'\|^2)$$

where $\gamma > 0$ is a tunable hyperparameter that controls how quickly similarity decays with distance.

### Single-Qubit Reduced Density Matrix (1-RDM)

For each qubit $k$, the three expectation values of the **Pauli observables** ($X$, $Y$, $Z$) combine into the single-qubit **reduced density matrix** (1-RDM):

$$\rho_k = \frac{1}{2}\bigl(I + \langle X \rangle \sigma_x + \langle Y \rangle \sigma_y + \langle Z \rangle \sigma_z\bigr)$$

Collecting these across all 60 qubits gives the 180-dimensional projected feature vector.

### The Projected Quantum Kernel

The PQK kernel between two data points is defined as:

$$k^{\textrm{PQ}}(x_i, x_j) = \exp\Big(-\gamma \sum_k \sum_{P \in \{X,Y,Z\}} \big(\textrm{Tr}[P\,\rho_k(x_i)] - \textrm{Tr}[P\,\rho_k(x_j)]\big)^2 \Big)$$

This is exactly an RBF kernel applied to the 180-dim projected feature vectors — which is why we can use scikit-learn's standard `SVC(kernel='rbf')` directly on the projection data.

### Geometric Difference

$$g_{cq} = \sqrt{\|\sqrt{K^q}\, \sqrt{K^c}\, (K^c + \lambda I)^{-2}\, \sqrt{K^c}\, \sqrt{K^q}\|_{\infty}}$$

where $K^c$ and $K^q$ are the classical and quantum-projected kernel matrices, and $\lambda$ is a regularization parameter.

> **Advanced note:** If $g$ is large, Huang et al. define additional quantities called model complexities ($s_c$, $s_q$) that further characterize the potential for quantum encoding benefit. See Huang et al., Section 4 and Supplementary Section 6 for details.

### Projection Column Layout

The 180 quantum projection features are organized in three blocks of 60 (one per measurement basis). Within each block, columns are ordered by qubit, which maps directly to motif positions:

| Position | ⟨X⟩ columns | ⟨Y⟩ columns | ⟨Z⟩ columns |
|----------|-------------|-------------|-------------|
| 1 | 0–14 | 60–74 | 120–134 |
| 2 | 15–29 | 75–89 | 135–149 |
| 3 | 30–44 | 90–104 | 150–164 |
| 4 | 45–59 | 105–119 | 165–179 |

Example: to extract all projection features for **position 1 only**, take columns `[0:15] + [60:75] + [120:135]` = 45 features.

### Note on Circuit Repetitions

The tutorial uses 24 repetitions of the ZZ Feature Map; the paper's main reported results used 8 repetitions in a separate experimental run. Your results may differ slightly from the paper's published numbers.

> **Why does this matter?** More repetitions generally improve measurement accuracy by reducing statistical noise, but increase computational cost. The choice of repetitions is a practical trade-off between precision and resource usage.

> **Tip:** Avoid going below 20% of the training set to maintain statistical validity. Set random seeds for reproducibility.


---

**Created by:** Sophy Shin

**Reviewed by:**  Meltem Tolunay, Olivia Lanes, Alexandre Choquette 



<div class="alert alert-info">

© IBM Corp., 2026

This is licensed under the Apache License, Version 2.0. You may obtain a copy of this license in the LICENSE file in the root directory of this source tree or at http://www.apache.org/licenses/LICENSE-2.0.

Any modifications or derivative works of this must retain this copyright notice, and modified files need to carry a notice indicating that they have been altered from the originals.
</div>